# 차트 복원 모형 실험: 국소 좌표에서 프레이즈 변주 생성하기

Methods 개정안(2026-11-11) 2.3–2.7절의 설정 (가)를 구현한 첫 실험입니다. 유형열 Y, 환입 R, 총음가 B를
고정하고, 유형 대표형에서의 편차(η)와 음가를 복원 모형 q_θ로 다시 뽑습니다.

이 Notebook이 답하려는 질문은 세 가지입니다.

1. **q_θ가 배우는 것이 있는가.** 학습에 쓰지 않은 마디의 (η, 음가)를 학습 없는 빈도표보다 잘 예측하는가.
2. **문맥이 주소로 쓰이는가.** 이웃 유형·경계 음을 주면 원곡 마디를 그대로 내는가(10월의 장·위치 암기와 같은 문제).
3. **학습 중 구조 손실(L_M, L_B)이 생성 시 거부를 줄이는가.** 줄인다면 변주의 폭은 유지되는가.

| 학습 조건 | 문맥 가림 | 구조 손실 |
|---|---|---|
| `rec` | 없음 | 없음 (L_rec만) |
| `rec_drop` | 이웃 유형·경계 음을 50% 확률로 가림 | 없음 |
| `struct` | 위와 같음 | L_M(유형 여유) + L_B(음가 합) |

**실행 순서**: Runtime → Change runtime type에서 **GPU**를 고른 뒤 위에서 아래로 실행합니다.
실행 방법과 결과 위치는 `docs/chart_model_colab.md`에 있습니다.

**덮어쓰기 없음**: 모든 결과는 `artifacts/chart_runs/<RUN_TAG>/` 아래에만 쓰입니다. 기존 MaskGIT 설정·체크포인트·
선율형(A–I)과 기존 `DiffDodree`, `DiffDodree_conddrop` clone은 건드리지 않고 `DiffDodree_chart`라는 별도 clone을 씁니다.

In [ ]:
# Cell 0 — 실험 전용 clone 준비 (Colab에서만 동작합니다)
# 기존 MyDrive/DiffDodree는 건드리지 않고, 실험 branch를 별도 폴더에 clone합니다.
from pathlib import Path
import importlib.util
import os
import subprocess

REPO_URL = "https://github.com/SHlee-TDA/DiffDodree.git"
BRANCH = "claude/method-section-model-improve-d5ad4d"
EXPERIMENT_ROOT = Path("/content/drive/MyDrive/DiffDodree_chart")

try:
    _in_colab = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    _in_colab = False

if _in_colab:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)

    def git(*args: str) -> str:
        return subprocess.check_output(["git", "-C", str(EXPERIMENT_ROOT), *args], text=True).strip()

    if not EXPERIMENT_ROOT.exists():
        subprocess.run(
            ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(EXPERIMENT_ROOT)],
            check=True,
        )
    else:
        if git("remote", "get-url", "origin") != REPO_URL:
            raise RuntimeError(f"{EXPERIMENT_ROOT}는 다른 저장소의 clone입니다.")
        if git("status", "--porcelain", "--untracked-files=no"):
            raise RuntimeError(
                f"{EXPERIMENT_ROOT}에 commit하지 않은 수정이 있습니다. 덮어쓰지 않도록 여기서 멈춥니다."
            )
        subprocess.run(["git", "-C", str(EXPERIMENT_ROOT), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(EXPERIMENT_ROOT), "checkout", BRANCH], check=True)
        subprocess.run(
            ["git", "-C", str(EXPERIMENT_ROOT), "merge", "--ff-only", f"origin/{BRANCH}"], check=True
        )
    print("Commit:", git("log", "-1", "--oneline"))
    os.environ["GUGAKGEN_PROJECT_ROOT"] = str(EXPERIMENT_ROOT)
else:
    print("Colab이 아니므로 현재 project folder를 그대로 사용합니다.")

In [ ]:
# Cell 1 — 공통 실행 환경 설정
# 이 Cell은 모든 tutorial Notebook의 첫 번째 Cell이며 여러 번 실행해도 안전합니다.

from pathlib import Path
import importlib.util
import os
import subprocess
import sys

# Google Drive의 project folder가 다르면 이 값만 수정합니다.
COLAB_PROJECT_ROOT = Path(
    "/content/drive/MyDrive/DiffDodree"
)

# 필요하면 환경변수로 project root를 직접 지정할 수 있습니다.
PROJECT_ROOT_OVERRIDE = os.environ.get("GUGAKGEN_PROJECT_ROOT")


def find_local_project_root(start: Path) -> Path:
    """현재 위치에서 pyproject.toml을 기준으로 project root를 찾습니다."""
    candidates = [start.resolve(), *start.resolve().parents]

    for candidate in candidates:
        if (candidate / "pyproject.toml").exists():
            return candidate

    raise FileNotFoundError(
        "project root를 찾지 못했습니다. "
        "GUGAKGEN_PROJECT_ROOT를 지정하거나 project folder에서 Notebook을 실행하세요."
    )


try:
    # 순수 local Python에는 상위 ``google`` namespace가 없을 수 있습니다.
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    # 이미 mount되어 있으면 기존 mount를 그대로 사용합니다.
    drive.mount("/content/drive", force_remount=False)

if PROJECT_ROOT_OVERRIDE:
    project_root = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
elif IN_COLAB:
    project_root = COLAB_PROJECT_ROOT.resolve()
else:
    project_root = find_local_project_root(Path.cwd())

if not project_root.exists():
    raise FileNotFoundError(
        f"project folder가 존재하지 않습니다: {project_root}\n"
        "COLAB_PROJECT_ROOT 또는 GUGAKGEN_PROJECT_ROOT를 확인하세요."
    )

bootstrap_script = project_root / "scripts" / "bootstrap_notebook.py"

if not bootstrap_script.exists():
    raise FileNotFoundError(
        f"Notebook bootstrap script를 찾지 못했습니다: {bootstrap_script}"
    )

environment_profile = "colab" if IN_COLAB else "local"

# 필요한 dependency만 확인하고 누락된 package만 설치합니다.
# Colab profile에서는 Colab이 제공한 CUDA용 PyTorch를 교체하지 않습니다.
subprocess.run(
    [
        sys.executable,
        str(bootstrap_script),
        "--project-root",
        str(project_root),
        "--profile",
        environment_profile,
        "--install-missing",
    ],
    check=True,
)

# 현재 kernel에서 src package를 즉시 import할 수 있게 합니다.
src_root = project_root / "src"

if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

os.environ["GUGAKGEN_PROJECT_ROOT"] = str(project_root)
os.chdir(project_root)

from gugakgen.device import resolve_device
from gugakgen.reproducibility import set_global_seed

runtime = resolve_device(preference="auto")
device = runtime.device

SEED = 2026
set_global_seed(SEED)

print("환경 설정 완료")
print(f"- 실행 환경: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print(f"- Project root: {project_root}")
print(f"- Device: {device}")
print(f"- AMP enabled: {runtime.amp_enabled}")
print(f"- Seed: {SEED}")

In [ ]:
# 실험 code가 들어 있는 commit인지, GPU가 잡혔는지 확인합니다.
import subprocess

import torch

from gugakgen.workflows import chart_experiment  # 없으면 branch가 잘못된 것입니다.

print("Commit:", subprocess.check_output(["git", "-C", str(project_root), "log", "-1", "--oneline"], text=True).strip())
print("Runtime:", runtime.as_dict())
if runtime.backend == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 연구자가 주로 바꾸는 값입니다. 학습 조건·step·Seed·가림 비율은 configs/chart_v0.yaml에 있습니다.
from datetime import datetime, timezone, timedelta

RUN_MODE = "full"  # 코드 경로만 확인하려면 "smoke" (30 step, Seed 1개, 2곡)

# None이면 실행 시각으로 새 이름을 만듭니다.
# 연결이 끊겨 다시 실행할 때는 이전 실행에서 출력된 RUN_TAG를 그대로 넣으세요.
# 끝난 단위(학습 1회, 생성 표 1개)는 건너뛰고 남은 것만 계산합니다.
RUN_TAG = None

CONFIG_PATH = project_root / "configs/chart_v0.yaml"

if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE는 'smoke' 또는 'full'이어야 합니다.")
if RUN_MODE == "full" and runtime.backend != "cuda":
    raise RuntimeError(
        "full 실행은 CUDA GPU에서만 합니다. Runtime → Change runtime type에서 GPU를 고르세요."
    )
if RUN_TAG is None:
    RUN_TAG = datetime.now(timezone(timedelta(hours=9))).strftime("%Y%m%d_%H%M")
print("RUN_TAG =", repr(RUN_TAG), " ← 다시 실행할 때 이 값을 쓰세요.")

In [ ]:
# 실험 전체를 실행합니다. 학습 조건마다
#   (1) 보류 평가: 마디 일부를 빼고 학습해, 뺀 마디의 예측 손실을 학습 길이별로 잽니다 (5묶음 × Seed 3개).
#   (2) 보류 손실이 가장 낮은 학습 길이로 전체 72마디를 학습하고 60곡씩 생성해 평가합니다.
# 마지막에 학습 없는 기준선(T0, T2, 빈도표 차트)을 같은 지표로 평가합니다.
# 작은 모형이라 T4에서 전체 20–40분 정도를 예상합니다(실측 전 추정). 생성·평가는 CPU에서 돕니다.
from gugakgen.workflows.chart_experiment import run_experiment

report = run_experiment(CONFIG_PATH, project_root, RUN_TAG, runtime, run_mode=RUN_MODE)
print("저장:", f"artifacts/chart_runs/{RUN_TAG}{'' if RUN_MODE == 'full' else '_smoke'}/report.json")

In [ ]:
# 0) 유형화 재현과 문맥의 주소성
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
typology = report["typology"]
print("유형열:", typology["sequence"])
print("대표 마디:", typology["medoid_measures"], "· 실루엣", round(typology["silhouette"], 3))
print("접촉:", typology["contact"])
audit = report["address_audit"]
print(
    f"문맥 κ=(앞 유형, 유형, 뒤 유형, 총음가)가 곡에서 한 자리만 가리키는 마디: {audit['kappa_unique_position']}/{audit['measures']}"
    f" · 한 가지 내용만 가리키는 마디: {audit['kappa_single_content']}/{audit['measures']}"
)

In [ ]:
# 1) 학습 길이에 따른 예측 손실 (음높이 + 음가, nat). 점선은 학습 없는 빈도표입니다.
import matplotlib.pyplot as plt
import numpy as np

from gugakgen.workflows.chart_experiment import curve_rows, generation_rows, heldout_rows, recall_rows

curves = pd.DataFrame(curve_rows(report))
reference = {
    name: np.mean([u["heldout"][name]["pitch_nll"] + u["heldout"][name]["duration_nll"] for u in report["heldout"]])
    for name in ("empirical_by_type", "empirical_global")
}
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for arm, frame in curves.groupby("arm"):
    axes[0].plot(frame["step"], frame["with_context"], marker="o", label=f"{arm} · held-out")
    axes[0].plot(frame["step"], frame["train"], linestyle=":", label=f"{arm} · train")
    axes[1].plot(frame["step"], frame["train_recall"], marker="o", label=arm)
for name, value in reference.items():
    axes[0].axhline(value, linestyle="--", color="gray", linewidth=1)
    axes[0].text(curves["step"].max(), value, f" {name}", va="bottom", ha="right", fontsize=8)
axes[0].set(xscale="log", xlabel="training step", ylabel="NLL per hidden event (pitch + duration)")
axes[1].set(xscale="log", xlabel="training step", ylabel="training phrases reproduced from a fully hidden input", ylim=(-0.02, 1.02))
for ax in axes:
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)
plt.show()
print("고른 학습 길이:", {arm: choice["selected"] for arm, choice in report["selected_steps"].items()})

In [ ]:
# 2) 보류 마디 예측 (고른 학습 길이에서). 낮을수록 좋습니다. 범위는 학습 Seed 사이의 최소–최대입니다.
display(pd.DataFrame(heldout_rows(report)).set_index(["arm", "predictor"]))

# 3) 전체 학습 모형의 암기 정도: 모든 음을 가린 입력에서 원곡 마디를 그대로 내는 비율
display(pd.DataFrame(recall_rows(report)).set_index(["arm", "condition"]))

In [ ]:
# 4) 생성 결과 (각 60곡). 학습 모형은 학습 Seed별 평균을 다시 평균한 값입니다.
display(pd.DataFrame(generation_rows(report)).set_index(["generator", "setting"]))

## 표 읽는 법

**1) 손실 곡선.** held-out 곡선이 내려갔다가 다시 오르는 지점부터 모형은 학습 마디를 외웁니다(오른쪽 그림의 재현율이 오름).
held-out 곡선의 최저점이 점선(빈도표)보다 낮아야 q_θ가 마디 사이에 통하는 규칙을 배웠다고 말할 수 있습니다.

**2) 보류 마디 예측.**
- `empirical_by_type` / `empirical_global`: 유형별 / 곡 전체의 η·음가 빈도표(문맥을 보지 않음). 학습 없는 기준입니다.
- `model` 대 `model_null_context`: 이웃 유형·경계 음이 예측에 보탬이 되는지를 봅니다.
- 보류 마디는 서로 다른 마디 42개(대표 마디 9개는 항상 학습에 포함)를 5묶음으로 나눈 것입니다. 유형화는 72마디 전체로
  했으므로, 보류 마디도 유형과 대표형을 정하는 데에는 쓰였습니다.

**3) 암기 정도.** `with_context`가 높고 `null_context`가 낮으면 문맥이 주소로 쓰이고 있다는 뜻입니다.
고른 학습 길이가 짧으면 둘 다 낮게 나옵니다.

**4) 생성 결과.**
- `setting`의 `rho`는 한 단계에서 가리는 사건의 비율, `no_rejection`은 유형 소속 거부를 끈 대조입니다.
- 구성으로 보장되는 값: `type_membership`(거부를 켠 경우 1), 환입 위치의 반복, 음가 합. 모형의 성과로 읽지 않습니다.
- `first_step_membership_rate`, `membership_rejection_rate`: 모형이 낸 후보가 스스로 유형 안에 머무는 비율과 거부율.
  `struct`가 `rec_drop`보다 거부율이 낮은지가 **학습 중 구조 손실의 효과**입니다.
- `own_medoid_distance`: 대표형까지의 평균 거리. 원곡 값은 0.39입니다(`report["typology"]`). 이보다 훨씬 작으면
  변주가 대표형으로 오그라든 것이고, 크면 유형의 가장자리로 밀린 것입니다.
- `original_phrase_rate`: 원곡에 있는 마디를 그대로 낸 비율. `unseen_note_transition_rate`, `interval_jsd`,
  `duration_jsd`, `unseen_boundary_rate`: 원곡에 없는 음 전이·경계 전이의 비율과 음정·음가 분포의 차이(낮을수록 원곡 양식에 가까움).
- 기준선: `T0_recombine`(같은 유형의 원곡 마디를 골라 붙임), `T2_edit`(T0에 이웃 음 치환·음가 교환),
  `chart_empirical`(같은 연쇄, q_θ 대신 유형별 빈도표). **학습 모형이 `chart_empirical`과 `T2_edit`보다 나은지**가 핵심 비교입니다.

음악적 품질은 이 표로 판단할 수 없습니다. 결과 폴더 경로(`DiffDodree_chart/artifacts/chart_runs/<RUN_TAG>/`)를 알려 주시면
Mac에서 읽어 해석하고, 들어 볼 악보를 뽑겠습니다.